# Unified inbox

What `read` fetches and what hooks receive, from every platform, in one SQLite file.

In [1]:
from amoorion import Amoorion

# What a platform's read would give.
from amoorion.models import Message

amo = Amoorion().use("dryrun")

# Publish…
amo.publish(text="Our release is out")

# …and read: reading keeps messages.
amo.read(limit=5)

# As hooks and other platforms would leave them:
# Messages from elsewhere, as if hooks and other platforms had delivered them.
amo.store.keep(
    [
        Message(
            platform="hook:github",
            text="me/tool: release published: v1.0",
            id="a",
        ),
        Message(
            platform="matrix",
            text="Is the RELEASE stable?",
            id="b",
        ),
        Message(
            platform="telegram",
            text="good morning",
            id="c",
        ),
    ]
)

# Newest first, whatever their origin.
[(message["platform"], message["text"]) for message in amo.inbox()]

[('telegram', 'good morning'),
 ('matrix', 'Is the RELEASE stable?'),
 ('hook:github', 'me/tool: release published: v1.0'),
 ('dryrun', 'Our release is out')]

One platform, or searching all of them:

In [2]:
# Only one platform.
[message["text"] for message in amo.inbox(platform="telegram")]

['good morning']

In [3]:
# By their text, across all of them.
[
    (message["platform"], message["text"])
    for message in amo.search(text="release")
]

[('matrix', 'Is the RELEASE stable?'),
 ('hook:github', 'me/tool: release published: v1.0'),
 ('dryrun', 'Our release is out')]

A message is kept once, by platform and id, so reading twice adds nothing.

In [4]:
# Reading the same messages again…
amo.read(limit=5)

# …adds no duplicates: a message is known by its platform and id.
len(amo.inbox())

4